# Lab 10: MLP Model for Time-Series Forecasting

**Student Name:** DANIYAL BASHARAT  
**Registration Number:** 22JZELE0467  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Lab Overview
This notebook develops a Multi-Layer Perceptron model for time-series forecasting. The code loads preprocessed time-series data, defines an MLP architecture, configures checkpoints and monitoring callbacks, trains the model, loads saved results, and evaluates prediction performance using regression metrics.

## Learning Objectives
- Set the working directory and import required machine learning, TensorFlow, and utility modules.
- Define an MLP neural network architecture for time-series input data.
- Configure optimizer, checkpoints, and training monitor callbacks.
- Load train, validation, and test CSV files for forecasting.
- Train and evaluate the model using MAE, MSE, RMSE, R2, and explained variance score.


## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 working path and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utilities.


In [2]:
import os
os.chdir(r'Z:\University\8th Semester\ML Lab\Lab 10,11')

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [6]:
model1 = MLP()
model1.summary()

c:\Users\engin\.conda\envs\ml\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        16,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,193 (63.25 KB)

 Trainable params: 16,193 (63.25 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and MLP Architecture
The following cells define time steps, number of features, and the MLP model structure used for forecasting.


In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [8]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'Z:\University\8th Semester\ML Lab\Lab 10,11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [9]:
os.path.exists(JSON_PATH)

True

In [10]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [11]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## Section 3: Checkpoints, Callbacks, and Training Setup
This section configures model checkpoints, training monitor paths, optimizer settings, and compile/training parameters.


In [12]:
import os
path_dataset =r'Z:\University\8th Semester\ML Lab\Lab 10,11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\engin\.conda\envs\ml\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [13]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.008354902267456055 sec


In [14]:
train_X.shape

(835, 24, 21)

In [15]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/60
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2505 - mae: 0.2505 - mape: 185.9495
Epoch 1: val_loss improved from None to 0.17662, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0001-loss0.18.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0001-loss0.18.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 0.1725 - mae: 0.1725 - mape: 103.4700 - val_loss: 0.1766 - val_mae: 0.1766 - val_mape: 60.9209
Epoch 2/60
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1091 - mae: 0.1091 - mape: 48.9882 
Epoch 2: val_loss did not improve from 0.17662
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0933 - mae: 0.0933 - mape: 45.6345 - val_loss: 0.1925 - val_mae: 0.1925 - val_mape: 64.4269
Epoch 3/60
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1299 - mae: 0.1299 - mape: 77.6427 
Epoch 3: val_loss improved from 0.17662 to 0.08281, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0003-loss0.08.h5



Epoch 3: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0003-loss0.08.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0935 - mae: 0.0935 - mape: 53.5256 - val_loss: 0.0828 - val_mae: 0.0828 - val_mape: 27.8955
Epoch 4/60
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0699 - mae: 0.0699 - mape: 34.8688 
Epoch 4: val_loss did not improve from 0.08281
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0662 - mae: 0.0662 - mape: 33.8602 - val_loss: 0.1380 - val_mae: 0.1380 - val_mape: 45.5894
Epoch 5/60
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0794 - mae: 0.0794 - mape: 41.2443 
Epoch 5: val_loss improved from 0.08281 to 0.08046, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0005-loss0.08.h5



Epoch 5: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0005-loss0.08.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0674 - mae: 0.0674 - mape: 32.8537 - val_loss: 0.0805 - val_mae: 0.0805 - val_mape: 29.0978
Epoch 6/60
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0623 - mae: 0.0623 - mape: 29.0610 
Epoch 6: val_loss improved from 0.08046 to 0.07312, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0006-loss0.07.h5



Epoch 6: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0006-loss0.07.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0636 - mae: 0.0636 - mape: 30.4067 - val_loss: 0.0731 - val_mae: 0.0731 - val_mape: 25.1309
Epoch 7/60
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0578 - mae: 0.0578 - mape: 27.1894 
Epoch 7: val_loss improved from 0.07312 to 0.05699, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0007-loss0.06.h5


...


Epoch 32: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0032-loss0.04.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0375 - mae: 0.0375 - mape: 16.1275 - val_loss: 0.0352 - val_mae: 0.0352 - val_mape: 10.1171
Epoch 33/60
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0336 - mae: 0.0336 - mape: 13.0107
Epoch 33: val_loss did not improve from 0.03524
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0338 - mae: 0.0338 - mape: 15.3297 - val_loss: 0.0609 - val_mae: 0.0609 - val_mape: 21.7696
Epoch 34/60
16/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0582 - mae: 0.0582 - mape: 27.7801 
Epoch 34: val_loss did not improve from 0.03524
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0426 - mae: 0.0426 - mape: 19.7218 - val_loss: 0.0471 - val_mae: 0.0471 - val_mape: 14.5803
Epoch 35/60
16/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0321 - mae: 0.0321 - mape: 18.4597 
Epoch 35: val_loss did not improve from 0.03524
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/st


Epoch 46: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0046-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0367 - mae: 0.0367 - mape: 16.9345 - val_loss: 0.0331 - val_mae: 0.0331 - val_mape: 11.0839
Epoch 47/60
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0318 - mae: 0.0318 - mape: 13.1422 
Epoch 47: val_loss did not improve from 0.03305
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0318 - mae: 0.0318 - mape: 17.4538 - val_loss: 0.0528 - val_mae: 0.0528 - val_mape: 18.4230
Epoch 48/60
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0358 - mae: 0.0358 - mape: 15.1980 
Epoch 48: val_loss did not improve from 0.03305
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0296 - mae: 0.0296 - mape: 13.0250 - val_loss: 0.0347 - val_mae: 0.0347 - val_mape: 10.5594
Epoch 49/60
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0291 - mae: 0.0291 - mape: 12.9753 
Epoch 49: val_loss improved from 0.03305 to 0.03287, saving model to Z:\University\8


Epoch 49: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0049-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0268 - mae: 0.0268 - mape: 13.5928 - val_loss: 0.0329 - val_mae: 0.0329 - val_mape: 10.2909
Epoch 50/60
15/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0316 - mae: 0.0316 - mape: 14.0452 
Epoch 50: val_loss did not improve from 0.03287
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0338 - mae: 0.0338 - mape: 16.0931 - val_loss: 0.0456 - val_mae: 0.0456 - val_mape: 14.6101
Epoch 51/60
16/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0359 - mae: 0.0359 - mape: 20.4897 
Epoch 51: val_loss did not improve from 0.03287
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0317 - mae: 0.0317 - mape: 15.9050 - val_loss: 0.0337 - val_mae: 0.0337 - val_mape: 10.3427
Epoch 52/60
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0302 - mae: 0.0302 - mape: 12.9561 
Epoch 52: val_loss did not improve from 0.03287
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/s


Epoch 54: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0054-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0245 - mae: 0.0245 - mape: 10.9755 - val_loss: 0.0307 - val_mae: 0.0307 - val_mape: 9.6576
Epoch 55/60
14/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0331 - mae: 0.0331 - mape: 14.6792 
Epoch 55: val_loss did not improve from 0.03065
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0292 - mae: 0.0292 - mape: 13.5813 - val_loss: 0.0408 - val_mae: 0.0408 - val_mape: 12.8188
Epoch 56/60
14/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0280 - mae: 0.0280 - mape: 13.0295
Epoch 56: val_loss did not improve from 0.03065
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0262 - mae: 0.0262 - mape: 13.5153 - val_loss: 0.0322 - val_mae: 0.0322 - val_mape: 10.5002
Epoch 57/60
16/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0272 - mae: 0.0272 - mape: 12.2910
Epoch 57: val_loss improved from 0.03065 to 0.02869, saving model to Z:\University\8th 


Epoch 57: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0057-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0260 - mae: 0.0260 - mape: 12.4049 - val_loss: 0.0287 - val_mae: 0.0287 - val_mape: 9.4046
Epoch 58/60
17/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0203 - mae: 0.0203 - mape: 8.5771 
Epoch 58: val_loss did not improve from 0.02869
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0220 - mae: 0.0220 - mape: 9.5416 - val_loss: 0.0385 - val_mae: 0.0385 - val_mape: 12.4735
Epoch 59/60
14/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0324 - mae: 0.0324 - mape: 14.9744 
Epoch 59: val_loss did not improve from 0.02869
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0305 - mae: 0.0305 - mape: 13.2235 - val_loss: 0.0417 - val_mae: 0.0417 - val_mape: 14.7932
Epoch 60/60
16/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0324 - mae: 0.0324 - mape: 16.3919 
Epoch 60: val_loss did not improve from 0.02869
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

In [16]:
model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0040-loss0.03.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
Mean Absolute Error (MAE): 5219.32
Median Absolute Error (MedAE): 5349.76
Mean Squared Error (MSE): 27717892.32
Root Mean Squared Error (RMSE): 5264.78
Mean Absolute Percentage Error (MAPE): 33.31 %
Median Absolute Percentage Error (MDAPE): 34.01 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Section 4: Dataset Loading and Forecast Evaluation
The remaining cells load CSV datasets, prepare forecasting windows, run prediction, and evaluate model performance using regression metrics.


# Fine Tuning

In [17]:
checkpoints = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0040-loss0.03.h5'
start_epoch= 56

In [18]:
# construct the callback to save only the best model to disk
EpochCheckpoint1 = ModelCheckpoint(
    checkpoints,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

TrainingMonitor1 = TrainingMonitor(
    FIG_PATH,
    jsonPath=JSON_PATH,
    startAt=start_epoch
)

callbacks = [EpochCheckpoint1, TrainingMonitor1]

model_path = r'Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0040-loss0.03.h5'
# model_path = None   # use this if you want to train from scratch

if model_path is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)

    opt = Adam(learning_rate=1e-3)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    print("[INFO] learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading Z:\University\8th Semester\ML Lab\Lab 10,11\E1-cp-0040-loss0.03.h5...
[INFO] learning rate: 9.999999747378752e-05


In [19]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0284 - mae: 0.0284 - mape: 12.9071
Epoch 1: val_loss improved from None to 0.02901, saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0001-loss0.03.h5



Epoch 1: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0001-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 0.0255 - mae: 0.0255 - mape: 12.1785 - val_loss: 0.0290 - val_mae: 0.0290 - val_mape: 8.4898
Epoch 2/10
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0224 - mae: 0.0224 - mape: 13.3147 
Epoch 2: val_loss did not improve from 0.02901
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0222 - mae: 0.0222 - mape: 11.0227 - val_loss: 0.0329 - val_mae: 0.0329 - val_mape: 9.9743
Epoch 3/10
20/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0211 - mae: 0.0211 - mape: 8.9078 
Epoch 3: val_loss did not improve from 0.02901
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0216 - mae: 0.0216 - mape: 10.0987 - val_loss: 0.0319 - val_mae: 0.0319 - val_mape: 9.6919
Epoch 4/10
19/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0223 - mae: 0.0223 - mape: 9.3712 
Epoch 4: val_loss did not improve from 0.02901
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 


Epoch 5: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0005-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0221 - mae: 0.0221 - mape: 10.2049 - val_loss: 0.0290 - val_mae: 0.0290 - val_mape: 8.5885
Epoch 6/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0217 - mae: 0.0217 - mape: 9.6941 
Epoch 6: val_loss did not improve from 0.02898
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0208 - mae: 0.0208 - mape: 9.5455 - val_loss: 0.0414 - val_mae: 0.0414 - val_mape: 13.2563
Epoch 7/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0262 - mae: 0.0262 - mape: 16.0295 
Epoch 7: val_loss did not improve from 0.02898
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0241 - mae: 0.0241 - mape: 12.1574 - val_loss: 0.0311 - val_mae: 0.0311 - val_mape: 9.5831
Epoch 8/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0195 - mae: 0.0195 - mape: 8.0374 
Epoch 8: val_loss improved from 0.02898 to 0.02624, saving model to Z:\University\8th Semester\


Epoch 8: finished saving model to Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0008-loss0.03.h5
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0211 - mae: 0.0211 - mape: 9.9717 - val_loss: 0.0262 - val_mae: 0.0262 - val_mape: 7.4957
Epoch 9/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0248 - mae: 0.0248 - mape: 11.3993 
Epoch 9: val_loss did not improve from 0.02624
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0235 - mae: 0.0235 - mape: 10.9401 - val_loss: 0.0394 - val_mae: 0.0394 - val_mape: 12.2222
Epoch 10/10
20/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0209 - mae: 0.0209 - mape: 10.4002
Epoch 10: val_loss did not improve from 0.02624
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0208 - mae: 0.0208 - mape: 10.6111 - val_loss: 0.0277 - val_mae: 0.0277 - val_mape: 7.7817


In [20]:
model = load_model(r'Z:\University\8th Semester\ML Lab\Lab 10,11\E2-cp-0006-loss0.03.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
Mean Absolute Error (MAE): 5593.31
Median Absolute Error (MedAE): 5731.58
Mean Squared Error (MSE): 31765447.43
Root Mean Squared Error (RMSE): 5636.08
Mean Absolute Percentage Error (MAPE): 35.7 %
Median Absolute Percentage Error (MDAPE): 36.25 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Final Conclusion
In this lab, an MLP neural network was implemented for time-series forecasting. The notebook demonstrates model creation, callback configuration, training workflow, and regression-based evaluation.

## Submitted By
**Student Name:** DANIYAL BASHARAT  
**Registration Number:** 22JZELE0467  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus
